In [20]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path

In [21]:
df = pd.read_csv('../data/data_banknote_authentication.txt', header=None)
print(df.head())
print(df.info())

         0       1       2        3  4
0  3.62160  8.6661 -2.8073 -0.44699  0
1  4.54590  8.1674 -2.4586 -1.46210  0
2  3.86600 -2.6383  1.9242  0.10645  0
3  3.45660  9.5228 -4.0112 -3.59440  0
4  0.32924 -4.4552  4.5718 -0.98880  0
<class 'pandas.DataFrame'>
RangeIndex: 1372 entries, 0 to 1371
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   0       1372 non-null   float64
 1   1       1372 non-null   float64
 2   2       1372 non-null   float64
 3   3       1372 non-null   float64
 4   4       1372 non-null   int64  
dtypes: float64(4), int64(1)
memory usage: 53.7 KB
None


In [22]:
df.columns = [
    "variance",
    "skewness",
    "curtosis",
    "entropy",
    "class"
]
df.head()

,variance,skewness,curtosis,entropy,class
0,3.62160,8.6661,-2.8073,-0.44699,0
1,4.54590,8.1674,-2.4586,-1.46210,0
2,3.86600,-2.6383,1.9242,0.10645,0
3,3.45660,9.5228,-4.0112,-3.59440,0
4,0.32924,-4.4552,4.5718,-0.98880,0


In [23]:
# class = 0 → genuine/authentic banknote
# class = 1 → forged/fake banknote

#output
# class
# 0    762
# 1    610
# Name: count, dtype: int64


In [24]:
df['class'].value_counts()
X = df.drop("class", axis=1)
# Now X dropped class column, axis - 1 is column 0 is row, contains the four measurements the model will use:
# variance
# skewness
# curtosis
# entropy
Y = df["class"]
# Y contains the correct answers: 0 or 1
# X = features/inputs
# y = target/answer
print(X.head())
print(Y.head())
print(X.shape)
print(Y.shape)

   variance  skewness  curtosis  entropy
0   3.62160    8.6661   -2.8073 -0.44699
1   4.54590    8.1674   -2.4586 -1.46210
2   3.86600   -2.6383    1.9242  0.10645
3   3.45660    9.5228   -4.0112 -3.59440
4   0.32924   -4.4552    4.5718 -0.98880
0    0
1    0
2    0
3    0
4    0
Name: class, dtype: int64
(1372, 4)
(1372,)


In [25]:
from sklearn.model_selection import train_test_split

In [26]:
X_train, X_test, Y_train, Y_test = train_test_split( #ensures the training and testing samples have the same proportion of class values as the dataset provided.
    X,
    Y,
    test_size=0.25,
    random_state=42, # hitchhikers guide to galaxy
    stratify=Y # preserves roughly the same proportion of genuine and fake notes in both sets.
    #stratify parameter in scikit-learn), it ensures that both sets retain the exact same percentage of 
    # target classes as the original dataset. This is crucial for handling imbalanced data, 
    # like credit card fraud datasets where fraudulent transactions are rare.
)

In [27]:
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", Y_train.shape)
print("y_test:", Y_test.shape)

X_train: (1029, 4)
X_test: (343, 4)
y_train: (1029,)
y_test: (343,)


In [29]:
#verify the split
print("Training features",X_train.shape)
print("Testing features",Y_train.shape)
print(X_train.value_counts(normalize=True))
print(Y_train.value_counts(normalize=True))


Training features (1029, 4)
Testing features (1029,)
variance  skewness   curtosis  entropy 
 0.37980   0.70980    0.75720  -0.44440    0.002915
 0.92970  -3.79710    4.64290  -0.29570    0.002915
 0.51950  -3.26330    3.08950  -0.98490    0.002915
-0.20620   9.22070   -3.70440  -6.81030    0.002915
-2.64790   10.13740  -1.33100  -5.47070    0.002915
                                             ...   
 0.54150   6.03190    1.68250  -0.46122    0.000972
-0.78690   9.56630   -3.78670  -7.50340    0.000972
 0.31803  -0.99326    1.09470   0.88619    0.000972
-0.27068   3.26740   -3.55620  -3.08880    0.000972
-0.78289   11.36030  -0.37644  -7.04950    0.000972
Name: proportion, Length: 1015, dtype: float64
class
0    0.554908
1    0.445092
Name: proportion, dtype: float64


In [31]:
from sklearn.dummy import DummyClassifier

In [33]:
baseline_model = DummyClassifier(
    strategy="most_frequent"
)

baseline_model.fit(X_train, Y_train)

baseline_accuracy = baseline_model.score(
    X_test,
    Y_test
)

print("Baseline accuracy:", baseline_accuracy) 

Baseline accuracy: 0.5568513119533528


In [36]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
logistic_model = make_pipeline(
    StandardScaler(),
    LogisticRegression()
)

logistic_model.fit(X_train, Y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('standardscaler', ...), ('logisticregression', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](4,)","['variance','skewness','curtosis','entropy']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,4
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
